In [1]:
import os

# Automatically gets the folder where your current notebook is running
current_dir = os.getcwd()
print("Your notebook is running in:", current_dir)

# This will ensure files save right here in this folder

Your notebook is running in: C:\Users\U S E R\Quant Edge


In [10]:
import numpy as np
import pandas as pd
import yfinance as yf

# Prescribed investable universe from QuantEdge Section 5
TICKERS = ["SPY", "EFA", "EEM", "AGG", "GLD", "XLE", "XLK"]

# Set start date to SPY's absolute earliest inception date
START_DATE = "1993-01-29"
END_DATE = "2026-10-20"

print("Fetching full historical data across all assets...")
raw_data = yf.download(
    TICKERS, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False
)

# Extract Adjusted Close prices (Section 9.2 governance compliance)
if isinstance(raw_data.columns, pd.MultiIndex):
  prices = raw_data["Adj Close"]
else:
  prices = raw_data[["Adj Close"]]
  prices.columns = TICKERS

# -------------------------------------------------------------
# 1. UNBALANCED PANEL (Full history per asset, preserves NaNs)
# -------------------------------------------------------------
prices.to_csv("portfolio_prices_unbalanced.csv")
print(
    "Saved 'portfolio_prices_unbalanced.csv' (contains all individual"
    " histories)."
)

# -------------------------------------------------------------
# 2. BALANCED JOINT PANEL (Clean matrix for portfolio optimization)
# -------------------------------------------------------------
# Drops rows prior to GLD's launch (2004-11-18) so all 7 assets are active simultaneously
prices_balanced = prices.dropna()
print(
    f"Balanced Joint Dataset Shape: {prices_balanced.shape} (Starts:"
    f" {prices_balanced.index.min().strftime('%Y-%m-%d')})"
)

# Compute log returns for statistical and risk modeling (Layer 2-4)
log_returns_balanced = np.log(prices_balanced / prices_balanced.shift(1)).dropna()

# Save balanced files for your Streamlit app and risk engine
prices_balanced.to_csv("portfolio_prices_balanced.csv")
log_returns_balanced.to_csv("portfolio_log_returns_balanced.csv")
print("Saved balanced price and log-return CSV files successfully!")

Fetching full historical data across all assets...
Saved 'portfolio_prices_unbalanced.csv' (contains all individual histories).
Balanced Joint Dataset Shape: (5513, 7) (Starts: 2004-11-18)
Saved balanced price and log-return CSV files successfully!


In [11]:
import datetime
import pandas as pd
import yfinance as yf

# Prescribed investable universe (Section 5)[cite: 5]
TICKERS = ["SPY", "EFA", "EEM", "AGG", "GLD", "XLE", "XLK"]
START_DATE = "1993-01-29"
END_DATE = "2026-10-20"

print("Downloading raw adjusted prices...")
raw_data = yf.download(
    TICKERS, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False
)

if isinstance(raw_data.columns, pd.MultiIndex):
  prices_raw = raw_data["Adj Close"]
else:
  prices_raw = raw_data[["Adj Close"]]
  prices_raw.columns = TICKERS

print(f"Raw data downloaded successfully. Shape: {prices_raw.shape}")
prices_raw.head()

Raw data downloaded successfully. Shape: (8488, 7)


Ticker,AGG,EEM,EFA,GLD,SPY,XLE,XLK
Date,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,24.053516,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,24.224611,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,24.275934,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,24.532562,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,24.635201,NaN,NaN


In [12]:
# Documenting metadata for your team report and reproducibility
download_date = datetime.datetime.now().strftime("%Y-%m-%d")

metadata = {
    "Data Source": "Yahoo Finance (via yfinance Python package)",
    "Download Date": download_date,
    "Price Type": (
        "Adjusted Close ('Adj Close') used to account for dividends and stock"
        " splits"
    ),
    "Investable Universe": TICKERS,
}

print("=== DATA GOVERNANCE METADATA ===")
for key, value in metadata.items():
  print(f"{key}: {value}")

=== DATA GOVERNANCE METADATA ===
Data Source: Yahoo Finance (via yfinance Python package)
Download Date: 2026-10-20
Price Type: Adjusted Close ('Adj Close') used to account for dividends and stock splits
Investable Universe: ['SPY', 'EFA', 'EEM', 'AGG', 'GLD', 'XLE', 'XLK']


In [13]:
print("Missing values per asset prior to calendar alignment:")
print(prices_raw.isnull().sum())

# Align calendars by dropping rows where any asset has a missing value (common sample)
prices_aligned = prices_raw.dropna()

print(f"\nCalendars aligned successfully!")
print(f"Common sample shape: {prices_aligned.shape}")
print(
    f"Common date range: {prices_aligned.index.min().strftime('%Y-%m-%d')} to"
    f" {prices_aligned.index.max().strftime('%Y-%m-%d')}"
)

Missing values per asset prior to calendar alignment:
Ticker
AGG    2687
EEM    2571
EFA    2165
GLD    2975
SPY       0
XLE    1490
XLK    1490
dtype: int64

Calendars aligned successfully!
Common sample shape: (5513, 7)
Common date range: 2004-11-18 to 2026-10-19


In [14]:
# Calculate daily log returns: r_t = ln(P_t / P_{t-1})[cite: 9]
log_returns = np.log(prices_aligned / prices_aligned.shift(1)).dropna()

print("Log returns calculated successfully.")
print(f"Log returns shape: {log_returns.shape}")
log_returns.head()

Log returns calculated successfully.
Log returns shape: (5512, 7)


Ticker,AGG,EEM,EFA,GLD,SPY,XLE,XLK
Date,,,,,,,
2004-11-19,-0.002046,-0.014400,-0.006495,0.008973,-0.011179,0.011890,-0.015586
2004-11-22,0.001559,0.003896,0.002928,0.003789,0.004758,0.015817,0.001428
2004-11-23,-0.000585,0.001732,0.001948,-0.004459,0.001525,0.006204,-0.004287
2004-11-24,0.000974,0.010590,0.004529,0.006682,0.002367,0.008301,0.008082
2004-11-26,-0.002924,0.016470,0.005793,0.005313,-0.000760,0.004258,-0.001422


In [15]:
# Compute Z-scores to flag extreme price movements (|Z-score| > 5)
mean_ret = log_returns.mean()
std_ret = log_returns.std()
z_scores = (log_returns - mean_ret) / std_ret

outlier_flags = abs(z_scores) > 5
outlier_counts = outlier_flags.sum()

print("Outlier count (|Z-score| > 5) per asset (Flagged, NOT deleted):")
print(outlier_counts)

Outlier count (|Z-score| > 5) per asset (Flagged, NOT deleted):
Ticker
AGG    17
EEM    21
EFA    24
GLD    11
SPY    21
XLE    20
XLK    14
dtype: int64


In [17]:
import os
from statsmodels.tsa.stattools import adfuller

# Ensure output folder exists
OUTPUT_DIR = "quantedge"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Run ADF stationarity test[cite: 9]
adf_results = {}
for ticker in log_returns.columns:
  adf_stat, p_value, _, _, crit_values, _ = adfuller(
      log_returns[ticker].dropna()
  )
  adf_results[ticker] = {
      "ADF Statistic": adf_stat,
      "p-value": p_value,
      "Critical Value (5%)": crit_values["5%"],
      "Stationary (p < 0.05)": p_value < 0.05,
  }

adf_summary_df = pd.DataFrame(adf_results).T
print("=== ADF Stationarity Test Summary ===")
print(adf_summary_df)

# Export clean datasets and audit reports
prices_path = os.path.join(OUTPUT_DIR, "portfolio_prices_clean.csv")
returns_path = os.path.join(OUTPUT_DIR, "portfolio_log_returns_clean.csv")
adf_path = os.path.join(OUTPUT_DIR, "stationarity_adf_results.csv")
outlier_path = os.path.join(OUTPUT_DIR, "outlier_flags_audit.csv")

prices_aligned.to_csv(prices_path)
log_returns.to_csv(returns_path)
adf_summary_df.to_csv(adf_path)
outlier_flags.to_csv(outlier_path)

print(f"\nAll clean datasets successfully exported to the '{OUTPUT_DIR}/' folder!")

=== ADF Stationarity Test Summary ===
    ADF Statistic p-value Critical Value (5%) Stationary (p < 0.05)
AGG    -15.816135     0.0           -2.862067                  True
EEM    -16.255032     0.0           -2.862066                  True
EFA    -16.217722     0.0           -2.862066                  True
GLD    -75.421346     0.0           -2.862065                  True
SPY    -16.706549     0.0           -2.862066                  True
XLE    -14.444406     0.0           -2.862067                  True
XLK    -18.463503     0.0           -2.862066                  True

All clean datasets successfully exported to the 'quantedge/' folder!


In [18]:
import pandas as pd

log_returns = pd.read_csv("quantedge/portfolio_log_returns_clean.csv", index_col=0, parse_dates=True)
print("Total missing values in clean log returns:", log_returns.isnull().sum().sum())

Total missing values in clean log returns: 0
